In [0]:
def run_autoloader(
    volume_path: str,
    source_subpath: str,
    table_name: str,
    file_pattern: str = "*", 
    file_format: str = "json",
    catalog: str = "workspace",
    schema: str = "default",
    infer_schema: bool = True,
    trigger_available_now: bool = True,
    overwrite: bool = False
):
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()
    
    # Base path do Volume
    base_path = f"/Volumes/{catalog}/{schema}/{volume_path}"
    
    # Caminhos internos
    source_path = f"{base_path}/{source_subpath}"
    checkpoint_path = f"{base_path}/checkpoints/{table_name}"
    schema_path = f"{base_path}/schemas/{table_name}"
    
    # Nome completo da tabela
    target_table = f"{catalog}.{schema}.{table_name}"

    # 🔄 LÓGICA DE OVERWRITE
    # inserir modo append
    if overwrite:
        print(f"Modo Overwrite ativado. Limpando metadados e tabela {target_table}...")
        # 1. Deleta a tabela do catálogo
        spark.sql(f"DROP TABLE IF EXISTS {target_table}")
        # 2. Limpa as pastas de checkpoint e schema no Volume para o stream recomeçar do zero
        dbutils.fs.rm(checkpoint_path, True)
        dbutils.fs.rm(schema_path, True)
    
    # Reader
    reader = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", file_format)
        .option("cloudFiles.schemaLocation", schema_path)
        .option("pathGlobFilter", file_pattern) # 👈 ISSO GARANTE A SEPARAÇÃO
    )
    
    if infer_schema:
        reader = reader.option("cloudFiles.inferColumnTypes", "true")
    
    df = reader.load(source_path)
    
    # Writer
    writer = (
        df.writeStream
        .format("delta")
        .option("checkpointLocation", checkpoint_path)
        .outputMode("append") # No stream para Delta, mantemos append após a limpeza inicial
    )
    
    if trigger_available_now:
        writer = writer.trigger(availableNow=True)
    
    query = writer.toTable(target_table)
    
    if trigger_available_now:
        query.awaitTermination()
    
    return f"Tabela {target_table} {'sobrescrita' if overwrite else 'atualizada'} com sucesso."